In [1]:
import random
import time
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from PIL import Image, ImageChops
import cv2

from sklearn.metrics import cohen_kappa_score
import timm

device = "cuda:0" if torch.cuda.is_available() else "cpu"




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [2]:
input_size = 384

transform = transforms.Compose(
    [
        transforms.Resize((input_size, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],  # ImageNet means
            std=[0.229, 0.224, 0.225],  # ImageNet stds
        ),
    ]
)

net = timm.create_model("tf_efficientnet_b4_ns", pretrained=True, num_classes=5)
net = net.to(device)

weight_path = "../input/weights/B4_3stage_13epoch_320.pkl"
if torch.cuda.is_available():
    map_loc = device
else:
    map_loc = torch.device("cpu")
try:
    net.load_state_dict(torch.load(weight_path, map_location=map_loc))
    print("Loaded custom weights.")
except FileNotFoundError:
    print("Custom weight file not found – using pretrained backbone with random head.")




/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Custom weight file not found – using pretrained backbone with random head.


In [3]:
# ---------- Minimal fine‑tuning ----------
class DRDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = f"{self.img_dir}/{row['id_code']}.png"
        img = Image.open(img_path).convert("RGB")
        img = self.transform(img)
        label = int(row["diagnosis"])
        return img, label


train_csv = "../input/aptos2019-blindness-detection/train.csv"
train_img_dir = "../input/aptos2019-blindness-detection/train_images"
train_dataset = DRDataset(train_csv, train_img_dir, transform)
train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True
)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=1e-4)

net.train()
epochs = 2  # small number of epochs for quick fine‑tuning
for epoch in range(epochs):
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs = imgs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = net(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{epochs}]  Loss: {epoch_loss:.4f}")

net.eval()
# -----------------------------------------

# ---------- Inference ----------
test_ids_df = pd.read_csv("../input/aptos2019-blindness-detection/test.csv")
test_ids = test_ids_df["id_code"].astype(str).values

submission = []
with torch.no_grad():
    for i, idx in enumerate(test_ids):
        print(f"Processing {i+1}/{len(test_ids)}: {idx}")
        image_path = f"../input/aptos2019-blindness-detection/test_images/{idx}.png"
        img = Image.open(image_path).convert("RGB")
        img = transform(img).unsqueeze(0).to(device)

        logits = net(img)  # shape: (1, 5)
        pred = torch.argmax(logits, dim=1).item()
        submission.append([idx, int(pred)])

submission = np.array(submission)
# -----------------------------------




Epoch [1/2]  Loss: 0.8239
Epoch [2/2]  Loss: 0.2895
Processing 1/367: b460ca9fa26f
Processing 2/367: 6cee2e148520
Processing 3/367: ca6842bfcbc9
Processing 4/367: 6cbc3dad809c
Processing 5/367: a9bc2f892cb3
Processing 6/367: 40e9b5630438
Processing 7/367: e4e343eaae2a
Processing 8/367: f4d3777f2710
Processing 9/367: 10a5026eb8e6
Processing 10/367: d8da9de62743
Processing 11/367: 9a3109657ac1
Processing 12/367: df0886f1e76b
Processing 13/367: 876deb29f000
Processing 14/367: 6987804eb464
Processing 15/367: 94a67ec0714f
Processing 16/367: f06e7a9df795
Processing 17/367: 6155cf375354
Processing 18/367: d74ccc796517
Processing 19/367: 44976c3b11a6
Processing 20/367: 4c5ab774a381
Processing 21/367: 4ec7796df40e
Processing 22/367: cc964bf04dbc
Processing 23/367: 28503940d10b
Processing 24/367: 77a9538b8362
Processing 25/367: a505981d1cab
Processing 26/367: fa3e544a7401
Processing 27/367: 308f7fce6f0d
Processing 28/367: d144144a2f3f
Processing 29/367: 0e0fc1d9810c
Processing 30/367: 1f3f32efaf

In [4]:
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
df.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")

Submission saved to submission.csv
